# 16 — Spain comparison

Analyse Portugal–Spain pre-tax price spreads and annual physical-balance differences. Spain is a comparison series, not automatically a valid causal control.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

import statsmodels.api as sm


In [ ]:
price_path = PATHS.interim / "weekly_oil_prices_tidy.csv"
if not price_path.exists():
    raise FileNotFoundError("Run the weekly price extraction before the Spain comparison.")
prices = pd.read_csv(price_path, parse_dates=["date"])
rows = []
for product in ["diesel", "gasoline"]:
    sub = prices.loc[prices["product"] == product]
    wide = sub.pivot(index="date", columns="country", values="price_without_tax_eur_per_1000l").dropna(subset=["PT", "ES"]).reset_index()
    wide["spread"] = wide["PT"] - wide["ES"]
    wide["post"] = (wide["date"] >= pd.Timestamp("2021-05-01")).astype(int)
    pre_mean = wide.loc[wide["post"] == 0, "spread"].mean()
    post_mean = wide.loc[wide["post"] == 1, "spread"].mean()
    rows.append({"product": product, "pre_spread_mean": pre_mean, "post_spread_mean": post_mean, "difference": post_mean - pre_mean, "unit": "EUR/1000L", "interpretation": "descriptive PT-ES spread change"})
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(wide["date"], wide["spread"])
    ax.axvline(pd.Timestamp("2021-05-01"), linestyle="--", linewidth=1)
    ax.axhline(0, linewidth=1)
    ax.set(title=f"Portugal minus Spain pre-tax {product} price", ylabel="EUR/1000L", xlabel="Date")
    fig.tight_layout()
    fig.savefig(PATHS.figures / f"pt_es_{product}_pretax_price_spread.png", dpi=180)
    plt.show()
spread_summary = pd.DataFrame(rows)
persist_dataframe(spread_summary, PATHS.metrics / "pt_es_price_spread_summary.csv")
display(spread_summary)


### Control validity checks for the report

Before using causal language, inspect pre-2021 trends, common tax changes, product-definition consistency, COVID-period distortions and the degree to which Spain experienced similar refinery/logistics shocks. If these diagnostics are weak, retain descriptive language.
